## Imports

In [2]:
import IPython.display as ipd
import json
import numpy as np
import matplotlib.pyplot as plt
import random
import soundfile as sf
import pandas as pd

import tensorflow as tf
import keras

## Carregando o modelo salvo

In [3]:
# Carregando o modelo salvo
model = keras.models.load_model('model_conv_1_sec_v1_0.keras')

# Carregando o scaler_y
import joblib
scaler_y = joblib.load('scaler_y_conv_1_sec_v1_0.save')

I0000 00:00:1758070577.408641   29459 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2756 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


## Carregando o dataset

In [4]:
## Carregando o dataset (lendo todos os arquivos .wav da pasta 'nsynth-test/audio')
import os
base_path = 'nsynth-test/audio'
file_list = [f for f in os.listdir(base_path) if f.endswith('.wav')]
samples = []
for file_name in file_list:
    signal = sf.read(os.path.join(base_path, file_name))
    samples.append(signal)

samples = pd.DataFrame(samples)
samples.head()

,0,1
0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",16000
1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",16000
2,"[0.0, 0.0, 0.0, -3.0517578125e-05, -9.15527343...",16000
3,"[0.0, 0.0, 9.1552734375e-05, 0.000244140625, 0...",16000
4,"[0.0, 0.0, 0.0, 0.0, 3.0517578125e-05, -3.0517...",16000


In [5]:
samples = samples.drop(columns=[1])
samples.head()

,0
0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"[0.0, 0.0, 0.0, -3.0517578125e-05, -9.15527343..."
3,"[0.0, 0.0, 9.1552734375e-05, 0.000244140625, 0..."
4,"[0.0, 0.0, 0.0, 0.0, 3.0517578125e-05, -3.0517..."


## Ajustando o formato do dataset

In [6]:
print(samples.shape)
x = np.array(samples[0].values.tolist())
print(x.shape)

(4096, 1)
(4096, 64000)


In [7]:
x = x.reshape((x.shape[0], x.shape[1], 1))
print(x.shape)

print(x[2])

(4096, 64000, 1)
[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]


## Chamando o modelo para predição do dataset ajustado

In [8]:
## Chamando o modelo para predição do dataset ajustado
y_pred_norm = model.predict(x)
y_pred = scaler_y.inverse_transform(y_pred_norm)

I0000 00:00:1758070584.038941   33473 service.cc:152] XLA service 0x7f29ac017740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758070584.038960   33473 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2025-09-16 21:56:24.061554: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1758070584.111847   33473 cuda_dnn.cc:529] Loaded cuDNN version 90300


 10/128 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step

I0000 00:00:1758070585.224844   33473 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step


In [9]:
y_pred = pd.DataFrame(y_pred)
y_pred.head()

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,1676.385254,0.692858,1.070072,1.439161,0.680620,0.993295,1.414693,0.661473,1.085746,1.282220,...,0.989243,1.694983,0.686668,0.968575,1.118138,0.693144,0.161784,0.427918,0.335684,0.369607
1,1591.174316,0.695141,1.071938,1.322761,0.676929,1.026084,1.505664,0.655820,1.082982,1.296634,...,0.963265,1.837769,0.696350,0.957697,1.208930,0.701044,0.100096,0.375683,0.201657,0.347048
2,586.566956,0.653471,1.554310,1.877068,0.670052,1.760860,1.948242,0.666880,1.929286,2.029264,...,1.705071,2.140307,0.640855,1.713265,1.603254,0.694854,0.504902,0.251579,0.472643,0.854601
3,1952.519775,0.682334,1.058701,1.323129,0.686287,1.065994,1.311532,0.667777,1.080073,1.345001,...,1.024058,1.313906,0.682536,1.047428,1.091400,0.689481,0.265898,0.340201,0.471101,0.591963
4,1595.606079,0.684915,1.176293,1.490362,0.680853,1.126056,1.427974,0.668053,1.143387,1.360550,...,1.134025,1.524263,0.680987,1.117855,1.196701,0.691734,0.220526,0.387573,0.443625,0.433454


In [ ]:
## Escrevendo os parâmetros preditos em arquivo JSON
params_list = y_pred.to_dict(orient="records")
with open("nsynth-pred/_params_pred.json", "w") as f:
    json.dump(params_list, f, indent=4)

# Ressintetizando o nsynth

## Chamando o sintetizador para cada parâmetro predito pelo modelo

In [10]:
# Convertendo o nome das colunas para os nomes dos parâmetros do sintetizador
column_names = ['frequencia_base', 'amplitude1', 'frequency1', 'beta2',
       'amplitude2', 'frequency2', 'beta3', 'amplitude3', 'frequency3',
       'beta4', 'amplitude4', 'frequency4', 'beta5', 'amplitude5',
       'frequency5', 'beta_carrier', 'amplitude_carrier', 'attack', 'decay',
       'sustain', 'release']
y_pred.columns = column_names
y_pred.head()

,frequencia_base,amplitude1,frequency1,beta2,amplitude2,frequency2,beta3,amplitude3,frequency3,beta4,...,frequency4,beta5,amplitude5,frequency5,beta_carrier,amplitude_carrier,attack,decay,sustain,release
0,1676.385254,0.692858,1.070072,1.439161,0.680620,0.993295,1.414693,0.661473,1.085746,1.282220,...,0.989243,1.694983,0.686668,0.968575,1.118138,0.693144,0.161784,0.427918,0.335684,0.369607
1,1591.174316,0.695141,1.071938,1.322761,0.676929,1.026084,1.505664,0.655820,1.082982,1.296634,...,0.963265,1.837769,0.696350,0.957697,1.208930,0.701044,0.100096,0.375683,0.201657,0.347048
2,586.566956,0.653471,1.554310,1.877068,0.670052,1.760860,1.948242,0.666880,1.929286,2.029264,...,1.705071,2.140307,0.640855,1.713265,1.603254,0.694854,0.504902,0.251579,0.472643,0.854601
3,1952.519775,0.682334,1.058701,1.323129,0.686287,1.065994,1.311532,0.667777,1.080073,1.345001,...,1.024058,1.313906,0.682536,1.047428,1.091400,0.689481,0.265898,0.340201,0.471101,0.591963
4,1595.606079,0.684915,1.176293,1.490362,0.680853,1.126056,1.427974,0.668053,1.143387,1.360550,...,1.134025,1.524263,0.680987,1.117855,1.196701,0.691734,0.220526,0.387573,0.443625,0.433454


In [11]:
# Liberando memória e GPU
import gc
gc.collect()
keras.backend.clear_session()

In [12]:
## Chamando o sintetizador para cada parâmetro predito pelo modelo
from fm_synth2 import FMSynth

# Iterando sobre todas as predições
for i in range(y_pred.shape[0]):
    # Extraindo os parâmetros preditos
    params = y_pred.iloc[i].to_dict()
    
    # # Criando uma instância do sintetizador FM
    fm_synth = FMSynth(
        amplitude1=params['amplitude1'],
        frequency1=params['frequency1'],
        beta2=params['beta2'],
        amplitude2=params['amplitude2'],
        frequency2=params['frequency2'],
        beta3=params['beta3'],
        amplitude3=params['amplitude3'],
        frequency3=params['frequency3'],
        beta4=params['beta4'],
        amplitude4=params['amplitude4'],
        frequency4=params['frequency4'],
        beta5=params['beta5'],
        amplitude5=params['amplitude5'],
        frequency5=params['frequency5'],
        beta_carrier=params['beta_carrier'],
        amplitude_carrier=params['amplitude_carrier'],
        attack=params['attack'],
        decay=params['decay'],
        sustain=params['sustain'],
        release=params['release'],
    )
    signal = fm_synth.synth_alg1(4, params["frequencia_base"])

    # Normalizando o pico
    peak = np.max(np.abs(signal))
    if peak > 0:
        signal = 0.891 * signal / peak

    # Salvando o áudio em arquivo
    sf.write(f"nsynth-pred/{file_list[i]}", signal, 16000)
